# Windows MT5 Market Data Export

Run this notebook on the Windows laptop where MetaTrader 5 is installed, open, and logged in to the broker account shown in Market Watch.

It exports OHLC candles for the exact 28 instruments in the supplied screenshot. Broker suffixes such as `.pc` and `.sc` are preserved. Output is compressed CSV plus a JSON manifest under `data/raw/mt5_export`.

> MT5 returns only history available to the terminal. In MT5, increase **Tools → Options → Charts → Max bars in chart** and allow the terminal to download history if an old start date returns fewer rows than expected.

## 1. Editable settings

Change `START_DATE` or `TIMEFRAMES` before running all cells. If multiple MT5 terminals are installed, set `MT5_TERMINAL_PATH` to the correct `terminal64.exe`; otherwise leave it as `None`.

In [ ]:
from __future__ import annotations

import json
import platform
import subprocess
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
OUTPUT_DIR = PROJECT_ROOT / "data" / "raw" / "mt5_export"

START_DATE = "2026-01-01"  # UTC, YYYY-MM-DD
TIMEFRAMES = ["M1", "M5", "M15", "H1", "H4", "D1"]
MT5_TERMINAL_PATH = None  # Example: r"C:\\Program Files\\Broker MT5\\terminal64.exe"

SYMBOLS = [
    "EURUSD", "GBPUSD", "USDJPY", "AUDUSD", "NZDUSD", "USDCAD",
    "XAUUSD.pc", "NAS100", "BTCUSD.sc", "USDCHF.pc", "GBPJPY.pc",
    "EURJPY.pc", "SP500", "AUDCAD.pc", "AUDCHF.pc", "AUDJPY.pc",
    "CADCHF.pc", "CADJPY.pc", "CHFJPY.pc", "COPPER-C", "EURAUD.pc",
    "EURCAD.pc", "EURCHF.pc", "EURGBP.pc", "GBPAUD.pc", "GBPCAD.pc",
    "GBPCHF.pc", "USOUSD.pc",
]

assert len(SYMBOLS) == 28 and len(set(SYMBOLS)) == 28
print(f"Project: {PROJECT_ROOT}")
print(f"Python:  {sys.executable}")
print(f"System:  {platform.platform()}")
print(f"Export:  {len(SYMBOLS)} symbols × {len(TIMEFRAMES)} timeframes")

## 2. Verify the MT5 connection and symbols

Install dependencies in the selected notebook kernel if needed:

```python
%pip install -r ../requirements-win-mt5.txt
```

Restart the kernel after installing `MetaTrader5`.

In [ ]:
try:
    import MetaTrader5 as mt5
except ImportError as exc:
    raise ImportError(
        "MetaTrader5 is not installed in this notebook kernel. Run "
        f"{sys.executable} -m pip install -r {PROJECT_ROOT / 'requirements-win-mt5.txt'}"
    ) from exc

init_kwargs = {"path": MT5_TERMINAL_PATH} if MT5_TERMINAL_PATH else {}
if not mt5.initialize(**init_kwargs):
    raise RuntimeError(f"MT5 initialize failed: {mt5.last_error()}")

try:
    account = mt5.account_info()
    terminal = mt5.terminal_info()
    if account is None:
        raise RuntimeError("MT5 is connected to a terminal but no trading account is logged in.")

    checks = []
    for symbol in SYMBOLS:
        info = mt5.symbol_info(symbol)
        selected = bool(info) and mt5.symbol_select(symbol, True)
        tick = mt5.symbol_info_tick(symbol) if selected else None
        checks.append({
            "symbol": symbol,
            "available": info is not None,
            "selected": selected,
            "digits": info.digits if info else None,
            "last_tick_utc": pd.to_datetime(tick.time, unit="s", utc=True) if tick else None,
        })

    symbol_check = pd.DataFrame(checks)
    print(f"Account: {account.login} | Server: {account.server}")
    print(f"Terminal: {terminal.path if terminal else 'unknown'}")
    display(symbol_check)

    missing = symbol_check.loc[~symbol_check["selected"], "symbol"].tolist()
    if missing:
        raise RuntimeError(
            "Unavailable MT5 symbols: " + ", ".join(missing) +
            ". Compare them with Market Watch and update SYMBOLS exactly, including suffixes."
        )
finally:
    mt5.shutdown()

## 3. Export historical candles

The exporter performs the same symbol preflight again, downloads each symbol/timeframe with `copy_rates_range`, normalizes timestamps to UTC, and writes one `.csv.gz` file per dataset.

In [ ]:
cmd = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "export_mt5_candles.py"),
    "--output-dir", str(OUTPUT_DIR),
    "--symbols", ",".join(SYMBOLS),
    "--timeframes", ",".join(TIMEFRAMES),
    "--start", START_DATE,
]
if MT5_TERMINAL_PATH:
    cmd.extend(["--terminal-path", MT5_TERMINAL_PATH])

print("Starting MT5 export...")
result = subprocess.run(cmd, cwd=PROJECT_ROOT)
if result.returncode != 0:
    raise RuntimeError(f"MT5 export failed with exit code {result.returncode}")
print(f"Export complete: {OUTPUT_DIR}")

## 4. Validate the export

In [ ]:
manifest_path = OUTPUT_DIR / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
summary = pd.DataFrame(manifest["files"])

expected_files = len(SYMBOLS) * len(TIMEFRAMES)
if len(summary) != expected_files:
    raise RuntimeError(f"Expected {expected_files} exports, found {len(summary)}")
if (summary["rows"] == 0).any():
    empty = summary.loc[summary["rows"] == 0, ["symbol", "timeframe"]]
    raise RuntimeError(f"Empty datasets found:\n{empty.to_string(index=False)}")

print(f"Range requested: {manifest['date_from']} → {manifest['date_to']}")
print(f"Files: {len(summary)} | Total candles: {manifest['total_rows']:,}")
display(summary.pivot(index="symbol", columns="timeframe", values="rows").fillna(0).astype(int))

## 5. Preview exported data

Columns are `broker_symbol`, `timeframe`, UTC `candle_time`, OHLC, `tick_volume`, broker spread, and `real_volume`. In spot FX, MT5 volume is commonly tick volume rather than centralized traded volume.

In [ ]:
sample_row = summary.query("symbol == 'EURUSD' and timeframe == 'M5'").iloc[0]
sample = pd.read_csv(sample_row["path"], parse_dates=["candle_time"])
print(sample_row["path"])
print(f"Rows: {len(sample):,} | {sample['candle_time'].min()} → {sample['candle_time'].max()}")
display(sample.tail(20))

## Output

Keep or copy the entire folder:

```text
data/raw/mt5_export
```

The compressed CSV files can be imported into the platform later with:

```bat
python scripts\import_candles_to_files.py data\raw\mt5_export
```